# 🧠 NVIDIA Nemotron-3-Nano-30B Supervised Fine-Tuning (SFT) Notebook

This notebook contains a complete **Supervised Fine-Tuning (SFT)** pipeline to train a LoRA adapter on the `train.csv` reasoning dataset. 

### 🏆 Key Features of this Training Pipeline:
1. **Template-Aware Tokenization**: Formats reasoning examples using standard chat structures via `tokenizer.apply_chat_template`.
2. **Target-Only Loss Masking**: Uses TRL's `DataCollatorForCompletionOnlyLM` to compute loss *only* on the model's answer, preventing training degradation from predicting prompt tokens.
3. **Parameter Efficient Fine-Tuning (PEFT)**: Sets up a rank 32 LoRA adapter targeting Nemotron projection layers.
4. **Kaggle GPU Optimized**: Configured for mixed-precision (`bfloat16`), gradient accumulation, and gradient checkpointing to fit training within standard Kaggle GPU limits (e.g., 2x T4 or L4/A100).
5. **Offline & Save Version Optimizations**: Configured to run offline without internet-based `pip` installations by loading pre-installed libraries and the NVIDIA Cutlass package path. Includes a `DRY_RUN` toggle to quickly save the notebook without running long training loops.

---

## 🛠️ Step 1: Environment Configuration & Import Dependencies

First, we load the required Python directories for NVIDIA utility scripts and import dependencies.

In [ ]:
import os
import site
import sys

# Add NVIDIA Cutlass DSL package path (needed offline on Kaggle to load Mamba kernels)
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)
    print("NVIDIA Cutlass DSL package path loaded successfully.")
else:
    print("NVIDIA Cutlass package path not found. If running locally, please ignore.")

In [ ]:
import subprocess
import torch
import polars as pl
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
import kagglehub

# Check CUDA
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# --- CONFIGURATION TOGGLES ---
# Set DRY_RUN = True to test notebook execution quickly (skips training & zipping)
# Set DRY_RUN = False to run full SFT training and output the final submission.zip
DRY_RUN = False

SHOULD_RUN_PIPELINE = not DRY_RUN
print(f"Should execute full training pipeline: {SHOULD_RUN_PIPELINE}")

## 📥 Step 2: Load and Format the Dataset

We load `train.csv` and format it. In this competition, the evaluation script extracts the answer from `\boxed{}`. Therefore, during SFT we must wrap our target answer in `\boxed{}`.

In [ ]:
# Load local data (make sure data path points to train.csv on Kaggle or local workspace)
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
df = pl.read_csv(data_path)
print(f"Loaded {len(df)} training examples.")

# Convert polars dataframe to Hugging Face dataset
dataset = Dataset.from_pandas(df.to_pandas())

def format_chat_template(example):
    """
    Constructs standard chat dialogues.
    We wrap the answer in \boxed{} as required by the competition evaluation metric.
    """
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": f"\\boxed{{{example['answer']}}}"}
    ]
    return {"messages": messages}

# Apply standard formatting
dataset = dataset.map(format_chat_template)
print("Sample Formatted Messages:")
print(dataset[0]["messages"])

## 🤖 Step 3: Load Model & Tokenizer

We download and load `NVIDIA Nemotron-3-Nano-30B-A3B-BF16`. We configure the model with `bfloat16` and gradient checkpointing to conserve memory.

In [ ]:
if SHOULD_RUN_PIPELINE:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    # Load tokenizer and set pad token
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        use_cache=False  # Must be False when using gradient checkpointing
    )
    model.gradient_checkpointing_enable()
    print("Model and tokenizer loaded successfully.")
else:
    print("Skipping model and tokenizer loading (DRY_RUN).")

## 📐 Step 4: Configure LoRA Adapter

We setup `peft.LoraConfig` matching the allowed constraints (rank <= 32) and target modules.

In [ ]:
if SHOULD_RUN_PIPELINE:
    LORA_RANK = 32

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=16,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("Skipping LoRA configuration (DRY_RUN).")

## 🏋️ Step 5: SFT Trainer Configuration

To prevent training on the prompt instructions, we use `DataCollatorForCompletionOnlyLM`. 
This detects the assistant response template token and masks everything before it in the loss calculation.

In [ ]:
if SHOULD_RUN_PIPELINE:
    # Define training parameters
    training_args = TrainingArguments(
        output_dir="./nemotron_adapter",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # Virtual batch size of 16
        learning_rate=1e-4,
        logging_steps=10,
        num_train_epochs=1,  # Adjust as needed (e.g. 1-3 epochs)
        bf16=True,
        optim="paged_adamw_8bit",  # Conserves VRAM
        save_strategy="no",        # Save manual adapter at the end
        report_to="none"
    )

    # Setup data collator to compute loss only on assistant outputs
    # Using Nemotron's assistant indicator token context
    response_template = "Assistant\n"
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer
    )

    # Initialize SFT Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=1024,  # Truncate long contexts to save memory
        data_collator=collator,
        args=training_args,
        peft_config=lora_config
    )

    print("Starting model training...")
    trainer.train()
    print("Training completed!")
else:
    print("Skipping SFT training execution (DRY_RUN).")

## 💾 Step 6: Save and Package Trained Adapter

Once training completes, we save the resulting adapter weights and config, and bundle them into the `submission.zip` package.

In [ ]:
if SHOULD_RUN_PIPELINE:
    OUTPUT_DIR = "/kaggle/working"
    print(f"Saving trained adapter to {OUTPUT_DIR}...")
    trainer.model.save_pretrained(OUTPUT_DIR)

    # Compress the adapter files directly at the root of the archive
    zip_cmd = "zip -j submission.zip /kaggle/working/adapter_config.json /kaggle/working/adapter_model.safetensors"
    print(f"Executing: {zip_cmd}")
    subprocess.run(zip_cmd, shell=True, check=True)
    print("Trained submission.zip packaged successfully!")
else:
    print("Skipping adapter saving and packaging (DRY_RUN).")